# unified-sbi scaling study on Colab

Runs `scripts/scaling_study.py`: NPE on the uncompressed data against the discovery + NPE pipeline, over total simulation budgets
`[250, 500, 1000, 2000, 5000, 10000]`, on the 8D Rosenbrock chain with a hidden 2D banana. Use a GPU runtime (*Runtime → Change runtime type*).

Results go to Google Drive, so a disconnected session can resume: finished (seed, budget) cells are skipped on rerun.

## 1. Get the code
Either clone your remote, or upload a zip of `unified-sbi` to Drive and unzip it. Edit whichever applies.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/unified-sbi"
# Option A: a git remote
# !git clone https://github.com/<you>/unified-sbi.git {REPO}
# Option B: a zip on Drive
# !unzip -q -o /content/drive/MyDrive/unified-sbi.zip -d /content/

In [ ]:
%cd {REPO}
%pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Smoke test (about 2 minutes)

In [ ]:
!python -m pytest -q
!python scripts/scaling_study.py --out /content/smoke --quick

## 3. The sweep

`--list-tasks` shows the grid. Running everything in one Colab session can take hours, so `TASKS` lets you run a slice per session (e.g. one seed at a time).
All cells write to the same `OUT`.

In [ ]:
OUT = "/content/drive/MyDrive/unified-sbi-results/scaling"
!python scripts/scaling_study.py --out {OUT} --list-tasks

In [ ]:
TASKS = range(0, 6)   # seed 0; use range(6, 12) and range(12, 18) for seeds 1 and 2
for task in TASKS:
    !python scripts/scaling_study.py --out {OUT} --task-id {task}

## 4. Scaling summary

In [ ]:
!python scripts/plot_scaling.py --results {OUT}
from IPython.display import Image, display
display(Image(f"{OUT}/figures/scaling.png"))
display(Image(f"{OUT}/figures/excess_power_law.png"))